> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 01 · Generative AI foundations

**CampusMate v0 (no code yet, just the brain)**  ·  ⏱️ Time budget: **25 min theory + 15 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Before we build an AI assistant for students, we need to know what the 'brain' inside it can and cannot do. Large language models are powerful, but they guess, they forget, and they cost money per word. Every design decision later in this workshop (tools, memory, guardrails, policies) exists to fix one of these weaknesses.

### What you'll do
- Learn what an LLM, a token, a context window and temperature are
- Watch a model confidently invent a CampusMate rule that does not exist (hallucination)
- Watch a model forget what you said one message ago (statelessness)

## Settings

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib

REGION = "us-east-1"                                 # the AWS region we use in this workshop
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite (CampusMate's brain)
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # same model for quick experiments

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"                             # the fictional campus data
print("Region:", REGION, "| Model:", MODEL_ID)

## 1. What is an LLM?

A **Large Language Model (LLM)** is a program trained on a huge amount of text to do one thing: **predict the next piece of text**.
That's it. Answering questions, writing code and summarising notes are all "predict the next word" done very well.

🏫 **Campus analogy:** an LLM is like a senior who has read every notice board, textbook and WhatsApp group on campus,
but never actually checked the official register. They answer instantly and sound sure, even when they are guessing.

## 2. Tokens

Models don't read words, they read **tokens**: chunks of text, roughly ¾ of an English word on average.
`"Attendance"` might be 2 tokens; `"AIDD303"` might be 3.

Why you care: **you pay per token** (input and output), and every model has a **limit** on tokens per request.

## 3. Context window

The **context window** is the maximum number of tokens the model can look at in one request: your question,
the instructions, any documents, and the answer it writes. Anything outside the window simply does not exist for the model.

🏫 Think of it as the size of the exam answer sheet. Run out of space and the rest is not marked.

## 4. Temperature

**Temperature** controls randomness when picking the next token. Low (0) = the safest, most likely word every time.
High (1) = more creative and varied.

Amazon Nova 2 Lite (our model) accepts `temperature`, so you can see the difference directly.

## 5. Hallucination

When the model doesn't know, it often **makes something up** that *sounds* right. This is called hallucination.
It's not lying. It's doing exactly what it was trained to do: produce likely-sounding text.

Fix (later today): give it **tools** that look up the real data, and tell it to use them.

## 6. Statelessness

Each request to a model is independent. The model **does not remember** your previous message unless *you send it again*.
Chat apps feel like they remember because they quietly resend the whole conversation every time.

Fix (Lab 6): **memory**, stored outside the model.

## 7. Thinking before answering

Newer models can **think** (reason privately) before they answer. Two consequences: thinking tokens are **billed like answer tokens**,
and if you allow only a tiny answer length, the model may use it all up thinking and say nothing. That's why we never set the limit too low.

## 🔬 Live check: talk to a model

To talk to a model we use **Strands Agents**, a small open-source Python library. Three lines: import it, create an agent with a model,
ask a question. (Notebook 03 explains Strands properly; today we just use it.)

In [ ]:
from strands import Agent

agent = Agent(model=CHEAP_MODEL_ID, callback_handler=None)   # callback_handler=None: stay quiet, we print the answer ourselves
print(agent("In one sentence, what is an AI agent?"))

## Demo A: hallucination

There is **no** 'Rule 42B' anywhere. Let's see what the model does when asked to quote it.

In [ ]:
agent = Agent(model=CHEAP_MODEL_ID, callback_handler=None)
print(agent("Quote CampusMate Rule 42B from the Jain AIDD student handbook about canteen tokens, word for word."))

Did it invent a rule, or did it admit it doesn't know? Newer models hedge more often, but **you can't rely on that**. Try rephrasing it as a confident fact ("As per Rule 42B, ...") and see.

## Demo B: statelessness

Two **separate** agents. The second one has never seen the first conversation.

In [ ]:
first = Agent(model=CHEAP_MODEL_ID, callback_handler=None)
print(first("Hi! My name is Priya and I'm in 3rd semester of AIDD."))
print("---")
second = Agent(model=CHEAP_MODEL_ID, callback_handler=None)     # a brand-new agent = a brand-new conversation
print(second("What is my name and which semester am I in?"))

Now ask the **first** agent again. It remembers, because the agent keeps the conversation in a list (`first.messages`)
and sends the whole list to the model every time. The model itself still remembers nothing.

In [ ]:
print(first("What is my name and which semester am I in?"))
print("Messages the agent is keeping:", len(first.messages))

## Demo C: temperature

Same question, temperature 0 vs 1, three times each (older model that still accepts temperature).

In [ ]:
from strands.models import BedrockModel

for t in (0.0, 1.0):
    names = []
    for _ in range(3):
        agent = Agent(model=BedrockModel(model_id=CHEAP_MODEL_ID, temperature=t), callback_handler=None)
        names.append(str(agent("Invent a 3-word name for a college tech fest. Reply with only the name.")).strip())
    print(f"temperature={t}:", names)

## ✍️ Your turn

Count tokens and money. Ask a fresh agent to explain *context window* to a first-year student, then work out the cost from the token counts (Nova 2 Lite: $0.33 per million input tokens, $2.75 per million output tokens).

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
MY_QUESTION = "TODO"   # TODO: write your question here

if MY_QUESTION != "TODO":
    from strands.models import BedrockModel
    agent = Agent(model=BedrockModel(model_id=CHEAP_MODEL_ID, streaming=False), callback_handler=None)   # streaming=False: token counts always come back
    result = agent(MY_QUESTION)
    print(result)
    usage = result.metrics.accumulated_usage          # how many tokens went in and came out
    cost = usage["inputTokens"] * 0.33 / 1_000_000 + usage["outputTokens"] * 2.75 / 1_000_000   # Nova 2 Lite prices
    print("tokens in:", usage["inputTokens"], "| tokens out:", usage["outputTokens"], f"| cost: ${cost:.6f}")
    # TODO: print the cost in rupees too (assume 1 USD = 84 INR, an approximate figure)
else:
    print("Skipped: fill in MY_QUESTION first.")

## ✅ Checkpoint

You are on track if:
- Demo A produced either an invented rule or a refusal, and you can explain why both are risky
- Demo B: the second agent didn't know Priya; the first agent did when asked again
- Temperature 0 gave (almost) the same name each time; 1.0 gave different names

**Stuck or behind?** Copy from `scripts/01_genai_demos.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `AccessDeniedException ... bedrock:InvokeModel` | Your IAM user lacks Bedrock permissions | Your IAM user needs the Labs 1-4 permissions listed in SETUP.md |
| `AccessDeniedException ... model access` | Nova 2 Lite isn't enabled in your account | Bedrock console → **Model access** → enable Amazon Nova 2 Lite |
| `NoCredentialsError` / `Unable to locate credentials` | AWS CLI not configured | Run `aws configure` in a terminal, then restart the kernel |
| `ModuleNotFoundError: No module named 'strands'` | The notebook isn't using your venv | Start Jupyter from the activated venv: `jupyter notebook notebooks/` |